In [140]:
import torch
import torch.nn.functional as F 
import matplotlib.pyplot as plt 

In [141]:
words = open("names.txt", "r").read().splitlines()
words[:3]

['emma', 'olivia', 'ava']

In [142]:
chars = sorted(list(set("".join(words))))

stoi = {s: i+1 for i, s in enumerate(chars)}
stoi["."] = 0 

itos = {i:s for s, i in stoi.items()}

In [143]:
# Build the dataset 
block_size = 3 
X, Y = [], [] 

for w in words:
   # print(w)

    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        #print("".join(itos[i] for i in context), "---->", itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [ ]:
# build the dataset and split into training, def/val and test sets 

def build_dataset(words):
    block_size = 3 
    X, Y = [], [] 

    for w in words:
        context = [0] * block_size 
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y



In [144]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [255]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [256]:
sum(p.nelement() for p in parameters) # total number of paramters 

3481

In [257]:
for p in parameters: 
    p.requires_grad = True 

In [258]:
# Since we have figured out the range where we think lies the optimal learning rate, we create a range of values between those two points 


# lr = torch.linspace(0.001, 1, steps=1000)

# This is a better way of re-implementing the code above 
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [282]:
lri = []
lossi = [] 

for i in range(10000):

    # mini-batch construct 
    ix = torch.randint(0, X.shape[0], (32,))

    # forward pass 
    # C[x[ix]] means we only want 32 rows of X which we will the be used to be indexed in C
    emb = C[X[ix]] # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
    logits = h @ W2 + b2 # (32, 27)


    # counts = logits.exp()
    # probs = counts / counts.sum(1, keepdims=True)
    # loss = -probs[torch.arange(32), Y].log().mean()

    # this one line of code is equivalent the the three commented lines above 
    # Y[ix] also select does same 32 rows corresponding Y label values 
    loss = F.cross_entropy(logits, Y[ix])
    #print(loss.item())

    # backward pass 
    for p in parameters:
        p.grad = None

    loss.backward() 
    # update 
    #lr = lrs[i]

    # main lr 
    #lr = 0.1

    # lr decay
    lr = 0.01

    for p in parameters: 
        p.data -= lr * p.grad 
    
    # track stats 
    #lri.append(lre[i])
    #lossi.append(loss.item())


In [283]:
loss

tensor(1.8435, grad_fn=<NllLossBackward0>)

In [284]:
emb = C[X[ix]] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Y[ix])
loss

tensor(1.8154, grad_fn=<NllLossBackward0>)

In [272]:
# plot lri and lossi 
# plt.plot(lri, lossi)

In [ ]:
# generate mini-batches 
# Select 32 training examples from X which will be 32 indexes 
torch.randint(0, X.shape[0], (32,))

tensor([ 29667, 156475, 121125,  64753,  32780,  25850,  80867, 152444,  99766,
         54264,  29515, 124541,  75444, 113338, 137159, 137205, 173523, 197197,
         66652,  24582, 122395,  44128, 184522, 211245, 177227,  60674, 175717,
         94333,  99417, 124749, 164267,  84410])

In [ ]:
# Splitting the dataset 

# training set - it is used to optimize the parameters of the model (like the model weights using gradient descent) - 80%
# dev/validation set - it is used to develop or train all the hyperparameters 
#                           - like the size of a hidden layer,
#                           - the size of the embedding, 
#                           - or the strength of regularization, etc 
# test set - It is used to evaluate the overall performance of the model at the end